# 148 라벨/비라벨 공통 센서이력 피처
같은설비·제품의센서스트림에서라벨여부와무관하게같은계산을수행. 품질라벨을피처에사용하지않는다. Raw값의교차제품/충돌/30분공백처리는145와동일. 모델검증구간은기존Run, 이력계산구간은기계제품연속관측구간으로구분.

In [ ]:
from pathlib import Path
import pandas as pd,numpy as np,json
root=Path.cwd();t=root/'reports/tables';ns={}
exec("from pathlib import Path\nimport os,sys,json,time\nimport numpy as np,pandas as pd\nfrom sklearn.metrics import confusion_matrix\nroot=Path.cwd();t=root/'reports/tables';outdir=root/'reports';data_root=root/'data'\nna={};nb={};nd={}\nexec(\"from pathlib import Path\\nimport os,sys,time\\nimport numpy as np,pandas as pd\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import RobustScaler\\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.svm import SVC\\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\\nfrom scipy.special import expit\\nroot=Path.cwd()\\nif not (root/'reports').exists():root=root.parent\\nt=root/'reports/tables'\\ntd=root/'data/taein'\\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\\nold=Path.cwd()\\ntry:\\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\\nfinally:os.chdir(old)\\nlab=D['lab'].copy()\\nlab=lab[lab['eval']].reset_index(drop=True)\\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\\nuse=lab.type.isin(['정상','미성형']).to_numpy()\\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\\nraw=lambda v:r[v+'_raw']\\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\\nphys=pd.DataFrame({\\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\\n 'recovery_dev':dev('Plasticizing_Time'),\\n 'backpressure_dev':dev('Average_Back_Pressure'),\\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\\n 'pressure_dev':dev('Max_Injection_Pressure'),\\n 'cushion_dev':dev('Cushion_Position')})\\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\\nfrom sklearn.cluster import KMeans\\nmemo={};audit=[]\\ndef predict(ref,q):\\n key=(tuple(ref),tuple(q))\\n if key in memo:return memo[key]\\n assert set(g[ref]).isdisjoint(g[q])\\n yy=y[ref];normal=yy==0;out={}\\n for view in ['std24']:\\n  X=views[view];imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\\n  full=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy)\\n  fullscore=expit(full.decision_function(b))\\n  pos=np.flatnonzero(yy==1);neg=np.flatnonzero(normal)\\n  k=min(2,len(np.unique(a[pos],axis=0)))\\n  modes=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(a[pos]) if k>1 else np.zeros(len(pos),int)\\n  experts=[]\\n  for mode in np.unique(modes):\\n   chosen=np.r_[neg,pos[modes==mode]]\\n   scores=[]\\n   for C in [.1,1.]:\\n    model=SVC(C=C,class_weight='balanced',gamma='scale').fit(a[chosen],yy[chosen])\\n    scores.append(expit(model.decision_function(b)))\\n   for C in [.1,1.]:\\n    model=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a[chosen],yy[chosen])\\n    scores.append(model.predict_proba(b)[:,1])\\n   experts.append(np.mean(scores,axis=0))\\n  expert=np.max(experts,axis=0)\\n  out[view+'_mode_max']=expert\\n  out[view+'_full']=fullscore\\n  out[view+'_geometric']=np.sqrt(expert*fullscore)\\n  out[view+'_agreement_min']=np.minimum(expert,fullscore)\\n  out[view+'_union_max']=np.maximum(expert,fullscore)\\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\\n return out\\ndef threshold(yy,s,limit=None):\\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\\n opts=[]\\n for j,c in enumerate(th):\\n  fp=int(((s>=c)&(yy==0)).sum())\\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\\n\",na)\nexec(\"from pathlib import Path\\nimport os,sys,time\\nimport numpy as np,pandas as pd\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import RobustScaler\\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.svm import SVC\\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\\nfrom scipy.special import expit\\nroot=Path.cwd()\\nif not (root/'reports').exists():root=root.parent\\nt=root/'reports/tables'\\ntd=root/'data/taein'\\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\\nold=Path.cwd()\\ntry:\\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\\nfinally:os.chdir(old)\\nlab=D['lab'].copy()\\nlab=lab[lab['eval']].reset_index(drop=True)\\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\\nuse=lab.type.isin(['정상','미성형']).to_numpy()\\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\\nraw=lambda v:r[v+'_raw']\\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\\nphys=pd.DataFrame({\\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\\n 'recovery_dev':dev('Plasticizing_Time'),\\n 'backpressure_dev':dev('Average_Back_Pressure'),\\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\\n 'pressure_dev':dev('Max_Injection_Pressure'),\\n 'cushion_dev':dev('Cushion_Position')})\\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\\nfrom sklearn.neighbors import NearestNeighbors\\nmemo={};audit=[]\\ncontext=lab[[v+'_raw' for v in ['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']]].to_numpy(float)\\ndef localize(ref,qq,X):\\n normal=ref[y[ref]==0]\\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[normal])\\n xx=imp.transform(X)\\n ci=SimpleImputer(strategy='median').fit(context[normal])\\n cx=ci.transform(context);cs=RobustScaler().fit(cx[normal]);cx=cs.transform(cx)\\n nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cx[normal])\\n near=nn.kneighbors(cx[qq],return_distance=False)\\n floor=np.maximum(xx[normal].std(axis=0)*.2,1e-6)\\n out=[]\\n for i,ids in zip(qq,near):\\n  peers=normal[ids];peers=peers[peers!=i][:30]\\n  out.append((xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor))\\n return np.asarray(out)\\ndef predict(ref,q):\\n key=(tuple(ref),tuple(q))\\n if key in memo:return memo[key]\\n assert set(g[ref]).isdisjoint(g[q])\\n yy=y[ref];normal=yy==0;out={}\\n for tag,X in [('localphysical12',views['physical12'])]:\\n  a=localize(ref,ref,X);b=localize(ref,q,X)\\n  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\\n  for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\\n   pp=[]\\n   for seed in [42,7]:\\n    m=cls(n_estimators=120,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\\n    pp.append(m.predict_proba(b)[:,1])\\n   out[tag+'_'+name]=np.mean(pp,axis=0)\\n  for C in [.1,1.]:\\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\\n   out[tag+'_svm'+str(C)]=expit(m.decision_function(b))\\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\\n  out[tag+'_lr']=m.predict_proba(b)[:,1]\\n  out[tag+'_mean']=np.mean([out[tag+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\\n return out\\ndef threshold(yy,s,limit=None):\\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\\n opts=[]\\n for j,c in enumerate(th):\\n  fp=int(((s>=c)&(yy==0)).sum())\\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\\n\",nb)\nexec(\"from pathlib import Path\\nimport os,sys,time\\nimport numpy as np,pandas as pd\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import RobustScaler\\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.svm import SVC\\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\\nfrom scipy.special import expit\\nroot=Path.cwd()\\nif not (root/'reports').exists():root=root.parent\\nt=root/'reports/tables'\\ntd=root/'data/taein'\\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\\nold=Path.cwd()\\ntry:\\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\\nfinally:os.chdir(old)\\nlab=D['lab'].copy()\\nlab=lab[lab['eval']].reset_index(drop=True)\\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\\nuse=lab.type.isin(['정상','미성형']).to_numpy()\\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\\nraw=lambda v:r[v+'_raw']\\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\\nphys=pd.DataFrame({\\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\\n 'recovery_dev':dev('Plasticizing_Time'),\\n 'backpressure_dev':dev('Average_Back_Pressure'),\\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\\n 'pressure_dev':dev('Max_Injection_Pressure'),\\n 'cushion_dev':dev('Cushion_Position')})\\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\\nval=lambda n:dense[n].to_numpy(float)\\nvs=lambda n:val(n+'__past10_std')\\nvd=lambda n:val(n+'__delta1')\\nve=lambda n:val(n+'__dev10')\\nthermal={\\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\\n}\\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\\n# Collapse several opposite-direction disturbances into shared intensity/count features.\\njointbase=np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Hopper_Temperature'),vs('Max_Injection_Speed'),vs('Cushion_Position'),vs('Plasticizing_Time'),vs('Barrel_Temperature_5'),vs('Average_Back_Pressure')])\\nmemo={}\\ndef predict(ref,q):\\n key=(tuple(ref),tuple(q))\\n if key in memo:return memo[key]\\n yy=y[ref];normal=yy==0\\n assert set(g[ref]).isdisjoint(g[q])\\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(jointbase[ref][normal])\\n a=imp.transform(jointbase[ref]);b=imp.transform(jointbase[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\\n def features(x):\\n  j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))\\n  return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])\\n model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(features(a),yy)\\n memo[key]=model.predict_proba(features(b))[:,1]\\n return memo[key]\\n\",nd)\nsourceC=\"from pathlib import Path\\nimport os,sys,time\\nimport numpy as np,pandas as pd\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import RobustScaler\\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.svm import SVC\\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\\nfrom scipy.special import expit\\nroot=Path.cwd()\\nif not (root/'reports').exists():root=root.parent\\nt=root/'reports/tables'\\ntd=root/'data/taein'\\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\\nold=Path.cwd()\\ntry:\\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\\nfinally:os.chdir(old)\\nlab=D['lab'].copy()\\nlab=lab[lab['eval']].reset_index(drop=True)\\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\\nuse=lab.type.isin(['정상','미성형']).to_numpy()\\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\\nraw=lambda v:r[v+'_raw']\\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\\nphys=pd.DataFrame({\\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\\n 'recovery_dev':dev('Plasticizing_Time'),\\n 'backpressure_dev':dev('Average_Back_Pressure'),\\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\\n 'pressure_dev':dev('Max_Injection_Pressure'),\\n 'cushion_dev':dev('Cushion_Position')})\\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\\nval=lambda n:dense[n].to_numpy(float)\\nvs=lambda n:val(n+'__past10_std')\\nvd=lambda n:val(n+'__delta1')\\nve=lambda n:val(n+'__dev10')\\nthermal={\\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\\n}\\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\\nmemo={};audit=[]\\ndef predict(ref,q):\\n key=(tuple(ref),tuple(q))\\n if key in memo:return memo[key]\\n assert set(g[ref]).isdisjoint(g[q])\\n yy=y[ref];normal=yy==0;out={}\\n for view,X in {'shock6':thermal['shock6']}.items():\\n  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal])\\n  a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\\n  for tag,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\\n   pp=[]\\n   for seed in [42,7]:\\n    m=cls(n_estimators=100,max_depth=4,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\\n    pp.append(m.predict_proba(b)[:,1])\\n   out[view+'_'+tag]=np.mean(pp,axis=0)\\n  for C in [.1,1.]:\\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\\n   out[view+'_svm'+str(C)]=expit(m.decision_function(b))\\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\\n  out[view+'_lr']=m.predict_proba(b)[:,1]\\n  out[view+'_mean']=np.mean([out[view+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\\n return out\\ndef threshold(yy,s,limit=None):\\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\\n opts=[]\\n for j,c in enumerate(th):\\n  fp=int(((s>=c)&(yy==0)).sum())\\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\\n\"\nnc=[]\nfor seeds in [[42,7],[11,23],[101,37]]:\n ns={};exec(sourceC.replace('for seed in [42,7]:','for seed in '+str(seeds)+':'),ns);nc.append(ns)\nlab=na['lab'];nlab=len(lab);yn=na['y'];gn=na['g'];usable=na['use']\nassert nlab==591 and int(yn[usable].sum())==8 and int(usable.sum())==574\nsensor_names=[x.replace('|std10','') for x in na['D']['names']]\nu_all=pd.read_csv(data_root/'original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])\ntarget_rows=u_all.loc[u_all.EQUIP_CD.eq('S14')&u_all.PART_NAME.str.contains('RH',na=False)].copy()\ntarget=target_rows.drop_duplicates('TimeStamp').sort_values('TimeStamp').reset_index(drop=True)\ntarget_raw_count=len(target_rows)\nrecords=[]\nknown_timestamps=set(pd.to_datetime(na['D']['lab'].TimeStamp))\nfor product in ['rg3','cn7']:\n for kind,priority in [('labeled',0),('unlabeled',1)]:\n  f=data_root/'original_split'/f'moldset_{kind}_{product}.csv'\n  raw=pd.read_csv(f,parse_dates=['TimeStamp'])\n  raw=raw[raw.EQUIP_CD.eq('S14')].copy()\n  raw['product']=product;raw['priority']=priority\n  records.append(raw[['TimeStamp','product','priority']+sensor_names])\n  if kind=='labeled' and product=='rg3':known_timestamps.update(raw.TimeStamp)\nstream=pd.concat(records,ignore_index=True)\nkeys=['product','TimeStamp']\nranges=stream.groupby(keys)[sensor_names].max()-stream.groupby(keys)[sensor_names].min()\nconflict=ranges.gt(1e-4).any(axis=1).rename('sensor_conflict')\nstream=stream.sort_values('priority').drop_duplicates(keys).merge(conflict,on=keys,how='left')\nknown=stream['product'].eq('rg3')&stream.TimeStamp.isin(known_timestamps)\nstream.loc[stream.sensor_conflict&~known,sensor_names]=np.nan\namb=stream.groupby('TimeStamp')['product'].transform('nunique').gt(1)\nambiguous_times=set(stream.loc[amb,'TimeStamp'])\nstream=stream[~amb].sort_values('TimeStamp').reset_index(drop=True)\ngap=stream.TimeStamp.diff().dt.total_seconds()\nboundary=gap.isna()|gap.gt(1800)|stream['product'].ne(stream['product'].shift())\nstream['segment']=boundary.cumsum()\nstream['run_id']='U30_'+stream.segment.astype(str).str.zfill(5)\nstream['history_count']=stream.groupby('segment').cumcount()\n# A uses current-inclusive std10; C/D dense history excludes the current shot, exactly as training.\nfor v in sensor_names:\n previous=stream.groupby('segment')[v].shift(1)\n stream[v+'__delta1']=stream[v]-previous\n stream[v+'__past10_std']=previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=5).std())\n med=previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=5).median())\n stream[v+'__dev10']=stream[v]-med\n stream[v+'__current_std10']=stream[v].groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=3).std())\n stream[v+'__sparse_dev']=stream[v]-previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=3).median())\nexcluded_overlap=int(target.TimeStamp.isin(known_timestamps).sum())\ntarget=target[~target.TimeStamp.isin(known_timestamps)].copy()\nexcluded_ambiguous=int(target.TimeStamp.isin(ambiguous_times).sum())\ntarget=target[~target.TimeStamp.isin(ambiguous_times)].copy()\nu=stream[stream['product'].eq('rg3')&stream.TimeStamp.isin(target.TimeStamp)].sort_values('TimeStamp').reset_index(drop=True)\nassert not u.TimeStamp.isin(known_timestamps).any() and u.TimeStamp.is_unique\nnu=len(u)\nstdU=u[[v+'__current_std10' for v in sensor_names]].to_numpy(float)\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphysU=np.column_stack([\n u.Max_Injection_Pressure-u.Max_Switch_Over_Pressure,\n ratio(u.Max_Switch_Over_Pressure,u.Max_Injection_Pressure),\n ratio(u.Filling_Time,u.Injection_Time),\n ratio(u.Filling_Time,u.Max_Injection_Speed),\n u.Plasticizing_Position-u.Cushion_Position,\n u.Plasticizing_Position__delta1-u.Cushion_Position__delta1,\n ratio(u.Plasticizing_Time,u.Cycle_Time),\n u.Plasticizing_Time__sparse_dev,\n u.Average_Back_Pressure__sparse_dev,\n u.Barrel_Temperature_5-(u.Mold_Temperature_3+u.Mold_Temperature_4)/2,\n u.Max_Injection_Pressure__sparse_dev,\n u.Cushion_Position__sparse_dev])\nshockU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Cushion_Position__past10_std])\njointU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Hopper_Temperature__past10_std,u.Max_Injection_Speed__past10_std,u.Cushion_Position__past10_std,u.Plasticizing_Time__past10_std,u.Barrel_Temperature_5__past10_std,u.Average_Back_Pressure__past10_std])\ncontextU=u[['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']].to_numpy(float)\n",ns)
stream=ns['stream'].copy();lab=ns['lab'];u=ns['u'];sensors=ns['sensor_names']
# This same history table/function supplies labeled and unlabeled; no sparse-label rolling.
for v in sensors:
 previous=stream.groupby('segment')[v].shift(1)
 stream[v+'__past30_std']=previous.groupby(stream.segment).transform(lambda s:s.rolling(30,min_periods=10).std())
 stream[v+'__past10_median']=previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=5).median())
cols=['TimeStamp','run_id','history_count','sensor_conflict']+sensors
for v in sensors:cols += [v+'__delta1',v+'__dev10',v+'__past10_std',v+'__current_std10',v+'__past30_std',v+'__past10_median']
common=stream[stream['product'].eq('rg3')][cols].copy()
assert common.TimeStamp.is_unique
l=lab[['TimeStamp','run_id','type','PassOrFail']].rename(columns={'run_id':'evaluation_run'}).merge(common,on='TimeStamp',validate='one_to_one')
assert len(l)==591 and l.type.eq('미성형').sum()==8
uu=u[['TimeStamp']].merge(common,on='TimeStamp',validate='one_to_one')
assert len(uu)==16733 and not uu.TimeStamp.isin(l.TimeStamp).any()
l.to_csv(t/'rg3_shortshot_148_common_labeled_features.csv',index=False,encoding='utf-8-sig')
uu.to_csv(t/'rg3_shortshot_148_common_unlabeled_features.csv',index=False,encoding='utf-8-sig')
# Audit current std10: formerly sparse labeled rows versus actual shared history.
old=ns['na']['views']['std24']
rows=[]
for j,v in enumerate(sensors):
 a=old[:,j];b=l[v+'__current_std10'].to_numpy()
 valid=np.isfinite(a)&np.isfinite(b)
 rows.append(dict(sensor=v,compared=int(valid.sum()),different=int((np.abs(a[valid]-b[valid])>1e-5).sum()),old_median=float(np.nanmedian(a)),shared_history_median=float(np.nanmedian(b))))
pd.DataFrame(rows).to_csv(t/'rg3_shortshot_148_window_change_audit.csv',index=False,encoding='utf-8-sig')
audit=dict(labeled_rows=len(l),normal=int(l.type.eq('정상').sum()),gas=int(l.type.eq('가스').sum()),shortshot=int(l.type.eq('미성형').sum()),unlabeled_rows=len(uu),label_overlap=0,
 history_gap_seconds=1800,history_reset='product change or >30min gap',current_std10='current inclusive rolling10,min3',
 past_std10='strict past rolling10,min5',past_std30='strict past rolling30,min10',dev10='current minus strict past10 median',
 previous_sensors_may_include_unlabeled=True,unlabeled_pseudo_labels=0,quality_label_used_in_feature_calculation=False)
(t/'rg3_shortshot_148_common_feature_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(audit,ensure_ascii=False))
print(pd.DataFrame(rows).sort_values('different',ascending=False).head(12).to_string(index=False))
